In [ ]:
#LOOKFORWARD YEARLY LOGISTIC ANALYSIS - COMPOSITE COGNITION 

"""
DYNAMIC look-forward landmark logistic (pooled composite cognition).

  Forward dynamic:     at horizon t, use the most recent visit whose status at
                       t is known, and ask whether dementia occurs within t
                       years of THAT visit.

At each forward horizon t (1..15 years):
  1. Classify every visit by its own lead time:
       case    : event == 1 and time_to_event <= t   (dementia within t of visit)
       control : time_to_event >= t                   (event-free through t)
       exclude : event == 0 and time_to_event < t     (status at t unknown)
  2. Among each person's classifiable (non-excluded) visits, take the MOST
     RECENT one (largest age_at_test) as the origin for that horizon. This uses
     the freshest assessment whose t-year outcome we can actually observe.
  3. Fit: dem_within_t ~ composite_z + age_at_test (logistic).
  4. Read absolute risk directly as the predicted probability at z = 0, 1, 2 SD
     worse, age at the sample mean.

One row per person per horizon, so no clustering. Every visit is used: a later
visit is the origin at short horizons (plenty of follow-up ahead), an earlier
visit becomes the origin at long horizons (only it has enough follow-up). This
is the dynamic-prediction analogue of the landmark Cox.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

#minimal plotting size
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

output_dir = 'lookforward_dynamic_int'
os.makedirs(output_dir, exist_ok=True)

data = pd.read_csv('cv_results_all_tests_merged_with_groups.csv')
data = data[data['test_name'] != 'PMem'].copy()

n_prev = (data['prevalent'] == 1).sum()
data = data[data['prevalent'] == 0].copy()
print(f"Dropped {n_prev:,} prevalent observations")

test_direction = {
    'PMat': -1, 'RT': -1, 'FI': 1, 'NM': 1, 'SD': 1, 'TMA': -1,
    'TMB': -1, 'MP': 1, 'PAL': 1, 'PV': 1, 'TR': 1, 'BL': 1
}

data['z_score_adj'] = -(data['z_score'] * data['test_name'].map(test_direction))

# administrative censoring age, derived only from people who reached it event-free
adm = (data.loc[data.event == 0, 'birth_decimal']
       + data.loc[data.event == 0, 'age_at_followup_end']).max()
data['potential_fu'] = (adm - data['birth_decimal']) - data['age_at_test']
print(f"Admin censoring date: {adm:.2f}; "
      f"median potential follow-up: {data['potential_fu'].median():.1f} y")

#Composite per visit (one row per person per visit)
data['visit_key'] = data['age_at_test'].round(3)
visit_df = (data.groupby(['eid', 'visit_key']).agg(
    z_score      = ('z_score_adj', 'mean'),
    age_at_test  = ('age_at_test', 'mean'),
    time_to_event= ('time_to_event', 'first'),
    event        = ('event', 'first'),
    potential_fu = ('potential_fu', 'first'),
).reset_index())
visit_df = visit_df[visit_df['time_to_event'] > 0].dropna(
    subset=['z_score', 'age_at_test', 'time_to_event', 'event']).copy()
visit_df['eid'] = visit_df['eid'].astype(str)

COMP_SD = visit_df['z_score'].std(ddof=1)
visit_df['z_score'] = visit_df['z_score'] / COMP_SD
print(f"Composite raw SD = {COMP_SD:.3f} (divided out; composite now SD 1)")

forward_windows = list(range(1, 21))

print(f"\nVisits = {len(visit_df):,}  |  participants = {visit_df['eid'].nunique():,}")
print(f"Median visits per person: {visit_df.groupby('eid').size().median():.0f}")

In [ ]:
#Dynamic look-forward loop in R
"""
-i visit_df -i forward_windows -o all_results -o risk_curves
Most-recent-classifiable-visit selection is done in R so the origin adapts to
each horizon. For each horizon t: classify visits, keep classifiable ones,
take each person's latest, fit logistic, read absolute risk.
"""
%load_ext rpy2.ipython

In [ ]:
%%R -i visit_df -i forward_windows -o all_results -o risk_curves

library(broom)
library(dplyr)

all_results <- list()
risk_curves <- list()
row_idx <- 1

for (t in forward_windows) {
  # Classify every visit at horizon t, drop the excluded (status unknown)
  classifiable <- visit_df %>%
    filter(potential_fu >= t) %>%
    group_by(eid) %>%
    slice_max(age_at_test, n = 1, with_ties = FALSE) %>%
    ungroup() %>%
    mutate(dem_within_t = case_when(
      event == 1 & time_to_event > (t - 1) & time_to_event <= t ~ 1,
      time_to_event >= t             ~ 0,
      TRUE                           ~ NA_real_
    )) %>%
    filter(!is.na(dem_within_t))

  n_cases <- sum(classifiable$dem_within_t)
  if (n_cases < 10 || nrow(classifiable) < 100) {
    cat("  t=", t, " SKIPPED (cases =", n_cases, ")\n"); next
  }

  fit <- tryCatch(
    glm(dem_within_t ~ z_score + age_at_test, data = classifiable,
        family = binomial(link = "logit")),
    error = function(e) return(NULL))
  if (is.null(fit)) next

  tidy_fit <- tidy(fit, conf.int = TRUE, exponentiate = TRUE)
  z_row <- tidy_fit[tidy_fit$term == "z_score", ]

  mean_age <- mean(classifiable$age_at_test)
  newd <- data.frame(z_score = c(0, 1, 2), age_at_test = mean_age)
  newd$predicted_risk <- predict(fit, newdata = newd, type = "response")
  newd$z_label <- c("average", "1 SD worse", "2 SD worse")
  newd$forward_window <- t
  risk_curves[[as.character(t)]] <- newd

  cat("  t=", t, " OR=", round(z_row$estimate, 3),
      " n=", nrow(classifiable), " incidence=",
      round(100 * n_cases / nrow(classifiable), 3), "%\n")

  all_results[[row_idx]] <- data.frame(
    forward_window = t, n_total = nrow(classifiable), n_cases = n_cases,
    incidence_pct = 100 * n_cases / nrow(classifiable),
    z_or = z_row$estimate, z_ci_low = z_row$conf.low,
    z_ci_high = z_row$conf.high, z_pval = z_row$p.value)
  row_idx <- row_idx + 1
}

all_results <- do.call(rbind, all_results)
risk_curves <- do.call(rbind, risk_curves)
cat("\nTotal models fit:", nrow(all_results), "\n")

In [ ]:
#COMBINED FIGURE: OR (relative) + absolute risk curves
all_results = all_results.sort_values('forward_window').reset_index(drop=True)
all_results.to_csv(f'{output_dir}/lookforward_results_dynamic.csv', index=False)
risk_curves.to_csv(f'{output_dir}/risk_curves_dynamic.csv', index=False)

risk_wide = risk_curves.pivot(index='forward_window', columns='z_label',
                              values='predicted_risk') * 100
col_order = ['average', '1 SD worse', '2 SD worse']
colors = {'average': 'black', '1 SD worse': 'darkorange', '2 SD worse': 'firebrick'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
ax.fill_between(all_results['forward_window'], all_results['z_ci_low'],
                all_results['z_ci_high'], alpha=0.25, color='black')
ax.plot(all_results['forward_window'], all_results['z_or'],
        marker='o', markersize=5, color='black', linewidth=2)
ax.axhline(y=1, color='darkgrey', linestyle='--', linewidth=1)
ax.set_xlabel('Years after cognitive test', fontsize=11)
ax.set_ylabel('OR per 1 SD worse cognition', fontsize=11)
ax.set_title('A. Relative odds (composite, dynamic origin)', fontsize=12)
ax.set_xticks(forward_windows)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
for col in col_order:
    ax.plot(risk_wide.index, risk_wide[col], marker='o', markersize=4,
            linewidth=2, color=colors[col], label=col)
ax.set_xlabel('Years after cognitive test', fontsize=11)
ax.set_ylabel('Predicted dementia probability (%)', fontsize=11)
ax.set_title('B. Absolute risk by cognition', fontsize=12)
ax.set_xticks(forward_windows)
ax.legend(loc='upper left', fontsize=9, frameon=False, title='Cognition')
ax.grid(alpha=0.3)

fig.suptitle('Dynamic look-forward (most-recent visit origin, all data): does cognition predict future dementia?',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{output_dir}/lookforward_combined_dynamic.png', dpi=800, bbox_inches='tight')
plt.show()